## 短期记忆
1.基于InMemorySaver存储短期记忆

In [3]:
from langchain.agents import create_agent
from anyio.lowlevel import checkpoint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

agent = create_agent(
    model = model,
    checkpointer = InMemorySaver(),
)

config = {"thread_id": "1"}

response = agent.invoke({"messages":[{"role":"user","content":"我是蔡徐坤"}]}, config=config)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

我是蔡徐坤
================================== Ai Message ==================================

您好！作为蔡徐坤的粉丝或关注者，您一定对他的音乐、舞台表现和时尚风格非常熟悉。他在音乐、舞蹈和创作方面展现出了独特的才华，也通过《偶像练习生》等节目积累了超高人气。无论是作为歌手还是舞台表演者，他都在不断突破自我，给观众带来惊喜。

如果您想聊聊他的音乐作品（比如《情人》《迷》）、舞台风格，或者近期动态，我很乐意和您一起分享和探讨～ 您有什么特别想聊的方向吗？ 😊


In [5]:
config = {"thread_id": "1"}

response = agent.invoke({"messages":[{"role":"user","content":"我是谁?"}]}, config=config)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

我是蔡徐坤
================================== Ai Message ==================================

您好！作为蔡徐坤的粉丝或关注者，您一定对他的音乐、舞台表现和时尚风格非常熟悉。他在音乐、舞蹈和创作方面展现出了独特的才华，也通过《偶像练习生》等节目积累了超高人气。无论是作为歌手还是舞台表演者，他都在不断突破自我，给观众带来惊喜。

如果您想聊聊他的音乐作品（比如《情人》《迷》）、舞台风格，或者近期动态，我很乐意和您一起分享和探讨～ 您有什么特别想聊的方向吗？ 😊
================================ Human Message =================================

我是谁
================================== Ai Message ==================================

哈哈，这个问题可难不倒我！如果从对话的上下文来看——您提到自己是“蔡徐坤”，那您可能是他本人，也可能是以他的粉丝身份在玩角色扮演，或者用这个称呼来表达对他的喜爱～ 

当然，如果您现在突然“失忆”了（笑），那我可以认真回答：**您是我正在对话的、独一无二的人类朋友**！无论您是谁，我都超级欢迎和您聊天～ 所以，需要我帮您“回忆”点什么，还是想聊聊别的呢？😉
================================ Human Message =================================

我是谁?
================================== Ai Message ==================================

哈哈，看来您是在和我玩一个“哲学小游戏”呢！如果从字面回答——**您是我正在对话的、有独立思考的鲜活人类**，而“蔡徐坤”这个名字，可能是您喜欢的偶像、您的网络昵称，或是您故意抛出的“烟雾弹”～ 

不过，如果这个问题关乎自我探索，那答案只有您自己能定义：您

2.基于postgresql持久化存储短期记忆


In [ ]:
import os
from langgraph.checkpoint.postgres import PostgresSaver
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(
    model = "deepseek:deepseek-v4-flash",
    extra_body = {"thinking":{"type":"disabled"}}
)

DB_URL = os.getenv("POSTGRES_DB_URL")

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    # 初始化数据库
    checkpointer.setup()

    agent = create_agent(
        model = model,
        checkpointer = checkpointer,
    )

    config = {"thread_id": "2"}

    print("="*10+"第一次调用"+("="*10))

    response = agent.invoke({"messages":[{"role":"user","content":"我是迪迦"}]}, config=config)

    print("="*10+"第二次调用"+("="*10))

    response = agent.invoke({"messages":[{"role":"user","content":"who am i"}]}, config=config)

    for msg in response["messages"]:
        msg.pretty_print()


总结：
- 1. InMemorySaver()将状态持久化到内存， 进程结束或重建Saver() 则历史状态丢失
- 2. 基于外部存储介质（如PostgreSQL）的持久化器，其存储的状态不会随进程终止而丢失，只要 不显式删除历史状态
